# NLP テキスト分類コンペ実践

TF-IDF ベースラインから BERT ファインチューニング、アンサンブルまで NLP コンペの流れを学びます。

> **Kaggle Tutorial** 対応ノートブック（`/competitions/nlp-text-classification`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# transformers の確認と、訓練・テスト分離済みのテキスト分類サンプルデータを準備します。
import os
import subprocess
import sys

try:
    import transformers  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)

import pandas as pd

def _ensure_nlp_comp_data():
    data_dir = os.path.abspath("./.kaggle_input/nlp-text-classification")
    os.makedirs(data_dir, exist_ok=True)
    orig_read_csv = pd.read_csv
    if not getattr(orig_read_csv, "_is_nlp_comp_wrapped", False):
        def _patched_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/<competition>/"):
                rel = filepath_or_buffer[len("/kaggle/input/<competition>/"):]
                filepath_or_buffer = os.path.join(data_dir, rel)
            return orig_read_csv(filepath_or_buffer, *args, **kwargs)
        _patched_read_csv._is_nlp_comp_wrapped = True
        pd.read_csv = _patched_read_csv

    pos_samples = [
        "Stratified cross validation kept our local score aligned with the leaderboard.",
        "Combining TF-IDF features with gradient boosting built a strong baseline.",
        "Careful feature engineering improved generalization on unseen test records.",
        "Ensembling diverse architectures reduced variance and stabilized predictions.",
    ] * 8
    neg_samples = [
        "Fitting the vectorizer before splitting leaked validation vocabulary.",
        "Overfitting to the public leaderboard caused a severe private shakeup.",
        "Missing value imputation across the full table distorted fold evaluation.",
        "Ignoring class imbalance degraded macro F1 on minority categories.",
    ] * 8

    train_df = pd.DataFrame({
        "text": pos_samples + neg_samples,
        "label": [1] * len(pos_samples) + [0] * len(neg_samples),
    })
    test_df = pd.DataFrame({
        "text": [
            "Out-of-fold validation prevented leakage and improved model blending.",
            "Target leakage inflated validation metrics and ruined test performance.",
            "Transfer learning with pretrained transformers boosted text accuracy.",
            "Unchecked distribution shift between train and test broke predictions.",
        ]
    })
    train_df.to_csv(os.path.join(data_dir, "train.csv"), index=False)
    test_df.to_csv(os.path.join(data_dir, "test.csv"), index=False)
    print(f"NLP competition dataset ready at: {data_dir}")

_ensure_nlp_comp_data()

## 概要

テキスト分類コンペでは、**軽い古典手法で強いベースライン**を作ってから、
**Transformer/BERT のファインチューニング**へ進むのが定石です。L7（NLP）の実践編です。

## 1. EDA とベースライン（TF-IDF + 線形）

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("/kaggle/input/<competition>/train.csv")  # text, label
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X = tfidf.fit_transform(df["text"])          # 訓練だけで fit
model = LogisticRegression(max_iter=1000).fit(X, df["label"])

まず TF-IDF + ロジスティック回帰で速い基準を作り、交差検証で評価します。

> **Warning:** `TfidfVectorizer` の `fit` は**訓練データだけ**に行い、検証/テストへは `transform`。 全体で fit すると語彙・IDF に検証側が混ざりデータリークになります。

## 2. BERT のファインチューニング（Hugging Face）

In [ ]:
# テストデータを読み込み、訓練データのみで fit した TF-IDF + 線形モデルでテスト予測確率を算出します
test_df = pd.read_csv("/kaggle/input/<competition>/test.csv")
pred_tfidf = model.predict_proba(tfidf.transform(test_df["text"]))[:, 1]

# 続く BERT トークナイズ用にクラス数とテスト文章リストを設定します
n_classes = int(df["label"].nunique())
texts = test_df["text"].tolist()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

name = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=n_classes)
enc = tok(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
# 小さい学習率(例 2e-5)・短いエポック・検証で早期終了

事前学習済み BERT を各タスクに転移学習します（L7-07）。GPU メモリに応じて
`max_length` とバッチサイズを調整します。

## 3. アンサンブル

In [ ]:
# BERT モデルでテスト文章の陽性クラス確率を推論します
import torch

model.eval()
with torch.no_grad():
    pred_bert = torch.softmax(model(**enc).logits, dim=-1)[:, 1].cpu().numpy()

In [ ]:
# 誤りの相関が低い TF-IDF線形 と BERT を加重平均
pred = 0.3 * pred_tfidf + 0.7 * pred_bert

In [ ]:
pd.DataFrame({"text": texts, "pred_tfidf": pred_tfidf.round(3), "pred_bert": pred_bert.round(3), "blend": pred.round(3)})

性質の異なる古典手法と BERT を ブレンディング すると、多様性が出て安定します（L6）。

> **Note:** 事前学習モデル・外部データには**ライセンスとコンペ規約**があります。利用可否を必ず確認しましょう （L6 / L8 の倫理・規約も参照）。

## 関連トラック

- L7-06 NLP I：前処理〜古典手法
- L7-07 NLP II：Transformer/BERT/Hugging Face
- L6 アンサンブル & コンペワークフロー